In [ ]:
!pip install transformers datasets gradio plotly pandas sentence-transformers -q

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re

raw = load_dataset("cardiffnlp/tweet_topic_multi", split="train_2020")
df = pd.DataFrame(raw)

# filter only health-related tweets using the label_name column
df["is_health"] = df["label_name"].apply(
    lambda labels: "fitness_&_health" in labels or "news_&_social_concern" in labels
)

df_health = df[df["is_health"]].copy()
df_other  = df[~df["is_health"]].sample(
    min(500, len(df[~df["is_health"]])), random_state=42
)

# combine: mostly health + small sample of other topics
# (keeps classifier honest — it still needs to detect "irrelevant")
df = pd.concat([df_health, df_other]).reset_index(drop=True)

print(f"Health tweets:     {len(df_health)}")
print(f"Non-health added:  {len(df_other)}")
print(f"Total:             {len(df)}")

df = df[["text", "date"]].dropna()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).reset_index(drop=True)

def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\{\{URL\}\}", "", text)
    text = re.sub(r"\{\{USERNAME\}\}", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(clean_text)
df = df[df["text"].str.len() > 10].reset_index(drop=True)

print(f"\nFinal dataset: {len(df)} tweets")
print(df["text"].head(5).tolist())

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(
    df["text"].tolist(),
    show_progress_bar=True,
    batch_size=128
)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(embeddings)

# inspect clusters to name them
print("--- Health subtopics discovered ---\n")
for cid in sorted(df["cluster"].unique()):
    count = len(df[df["cluster"] == cid])
    samples = df[df["cluster"] == cid]["text"].head(3).tolist()
    print(f"Cluster {cid} ({count} claims):")
    for s in samples:
        print(f"   {s[:100]}")
    print()

In [ ]:
CLUSTER_NAMES = {
    0: "general news & social concern",
    1: "sports & accidents",
    2: "fitness & exercise",
    3: "protests & climate",
    4: "politics & government",
    5: "mental health & celebrity",
    6: "climate change",
    7: "vaccines & flu",
    8: "environment & pollution",
    9: "social media & health events"
}

df["subtopic"] = df["cluster"].map(CLUSTER_NAMES)
print(df["subtopic"].value_counts())

In [ ]:
from transformers import pipeline
from tqdm.auto import tqdm

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0
)

LABELS = [
    "symptom report",
    "prevention behavior",
    "misinformation",
    "news or official report",
    "irrelevant"
]

results = []
BATCH = 32

for i in tqdm(range(0, len(df), BATCH)):
    batch = df["text"].iloc[i:i+BATCH].tolist()
    out = classifier(batch, LABELS, multi_label=False)
    for o in out:
        results.append({
            "label": o["labels"][0],
            "score": round(o["scores"][0], 3)
        })

df["label"]      = [r["label"] for r in results]
df["confidence"] = [r["score"]  for r in results]

print(df["label"].value_counts())
print(df[["text","subtopic","label","confidence"]].head(5))

In [ ]:
import numpy as np

df["week"] = df["date"].dt.to_period("W")
weekly = (df.groupby(["week", "label"])
            .size()
            .unstack(fill_value=0)
            .reset_index())

if "symptom report" not in weekly.columns:
    weekly["symptom report"] = 0

weekly = weekly.sort_values("week")

symptom_counts = weekly["symptom report"].values
signal = []
for i in range(len(symptom_counts)):
    if i < 3:
        signal.append(1.0)
    else:
        baseline = np.mean(symptom_counts[i-3:i])
        ratio = symptom_counts[i] / baseline if baseline > 0 else 1.0
        signal.append(round(ratio, 3))

weekly["outbreak_signal"] = signal
print(weekly[["week","symptom report","outbreak_signal"]].tail(10))

In [ ]:
low_conf = (df[df["confidence"] < 0.5]
            .sort_values("confidence")
            [["text","subtopic","label","confidence"]]
            .head(50)
            .reset_index(drop=True))

print(f"Low-confidence predictions: {len(low_conf)}")
print(low_conf.head(5))

In [ ]:
import gradio as gr
import plotly.graph_objects as go

def make_trend_chart():
    fig = go.Figure()
    label_cols = [c for c in weekly.columns
                  if c not in ["week","outbreak_signal"]]
    colors = {
        "symptom report":          "#E24B4A",
        "prevention behavior":     "#1D9E75",
        "misinformation":          "#BA7517",
        "news or official report": "#378ADD",
        "irrelevant":              "#888780"
    }
    for lbl in label_cols:
        fig.add_trace(go.Scatter(
            x=weekly["week"].astype(str),
            y=weekly[lbl],
            name=lbl,
            line=dict(color=colors.get(lbl,"#888780"), width=2)
        ))
    fig.add_trace(go.Scatter(
        x=weekly["week"].astype(str),
        y=weekly["outbreak_signal"] * 10,
        name="outbreak signal x10",
        line=dict(color="#7F77DD", width=2, dash="dash")
    ))
    fig.update_layout(
        title="Weekly health signal trends",
        xaxis_title="Week", yaxis_title="Count",
        plot_bgcolor="white", paper_bgcolor="white",
        legend=dict(orientation="h", y=-0.2)
    )
    return fig

def make_subtopic_chart():
    counts = df["subtopic"].value_counts()
    fig = go.Figure(go.Bar(
        x=counts.index.tolist(),
        y=counts.values.tolist(),
        marker_color="#7F77DD"
    ))
    fig.update_layout(
        title="Health subtopics discovered by K-means",
        xaxis_title="Subtopic", yaxis_title="Count",
        plot_bgcolor="white", paper_bgcolor="white"
    )
    return fig

def classify_text(text):
    if not text.strip():
        return "Please enter a health claim.", 0.0, "unknown"
    result = classifier(text, LABELS, multi_label=False)
    label  = result["labels"][0]
    score  = round(result["scores"][0], 3)
    emb    = embedder.encode([text])
    cluster_id = kmeans.predict(emb)[0]
    subtopic = CLUSTER_NAMES.get(cluster_id, "general health")
    return label, score, subtopic

with gr.Blocks(title="Public Health Signal Detector") as demo:
    gr.Markdown("## Public Health Signal Detector")
    gr.Markdown(
        "Classifies health claims into epidemiological categories "
        "and discovers hidden health subtopics using unsupervised learning."
    )
    with gr.Tabs():
        with gr.Tab("Live classifier"):
            inp = gr.Textbox(
                label="Enter a health claim",
                placeholder="e.g. The vaccine causes myocarditis in young men"
            )
            btn = gr.Button("Classify")
            out_label    = gr.Label(label="Category")
            out_conf     = gr.Number(label="Confidence")
            out_subtopic = gr.Textbox(label="Health subtopic")
            btn.click(classify_text,
                      inputs=inp,
                      outputs=[out_label, out_conf, out_subtopic])

        with gr.Tab("Trend chart"):
            gr.Plot(value=make_trend_chart())

        with gr.Tab("Subtopic map"):
            gr.Plot(value=make_subtopic_chart())

        with gr.Tab("Error analysis"):
            gr.Markdown("### Low-confidence predictions")
            gr.Dataframe(value=low_conf)

demo.launch(share=True)

In [ ]:
df.to_csv("tweets_classified.csv", index=False)
weekly.to_csv("weekly_trends.csv", index=False)
low_conf.to_csv("low_conf.csv", index=False)
print("saved!")

In [ ]:
# Kaggle version — files appear in the output panel on the right
# click Output on the right sidebar to find and download them
import shutil
shutil.copy("tweets_classified.csv", "/kaggle/working/tweets_classified.csv")
shutil.copy("weekly_trends.csv", "/kaggle/working/weekly_trends.csv")
shutil.copy("low_conf.csv", "/kaggle/working/low_conf.csv")
print("Files ready in /kaggle/working/ — check Output panel on the right")